Import `pandas` with placeholder `pd` to make use of DataFrames to hold values for our variables.

In [1]:
import pandas as pd

Import `numpy` with placeholder `np` since our variables are stored as arrays.

In [2]:
import numpy as np

Import `Path` from `pathlib` to handle file paths cleanly without needing `os` or `glob`.

In [3]:
from pathlib import Path

We set up the file paths pointing to the raw survey files, the GDP dataset, and the output folder.

In [4]:
BASE = Path.cwd().parent / 'ph-lfs-stagnation'
if not (BASE / 'data').exists():
    BASE = Path.cwd().parent
if not (BASE / 'data').exists():
    BASE = Path.cwd()

DATA_DIR = BASE / 'data' / 'PHL-PSA-LFS-2021-2024-PUF'
GDP_PATH = (BASE / 'data' / 'gdp-per-person-employed-constant-ppp'
                  / 'gdp-per-person-employed-constant-ppp.csv')
OUT_DIR  = BASE / 'data'

print('BASE:', BASE)
print('DATA_DIR exists:', DATA_DIR.exists())

BASE: c:\Users\clyde\Desktop\ph-lfs-stagnation
DATA_DIR exists: True


## 1. Schema Harmonisation

Three survey schemas span the 2021–2024 data:

| Schema | Months | Key difference |
|--------|--------|----------------|
| A | Jan–Jul 2021 | Standard PSA LFS columns |
| B | Aug–Dec 2021 | Re-numbered HHMEM format (questionnaire redesign) |
| C | 2022–2024 | Standard columns + extra fields (arrangement, province, ethnicity) |

We rename each to a unified column set, keeping only the variables needed for target construction and prediction.

In [5]:
SCHEMA_A = {
    'PUFREG':        'region',
    'PUFURB2015':    'urban_rural',
    'PUFSVYMO':      'survey_month',
    'PUFSVYYR':      'survey_year',
    'PUFHHSIZE':     'hh_size',
    'PUFC01_LNO':    'line_no',
    'PUFC03_REL':    'relationship',
    'PUFC04_SEX':    'sex',
    'PUFC05_AGE':    'age',
    'PUFC06_MSTAT':  'marital_status',
    'PUFC07_GRADE':  'education_grade',
    'PUFC10_CONWR':  'currently_working',
    'PUFC11_WORK':   'worked_last_week',
    'PUFC14_PROCC':  'occupation_code',
    'PUFC16_PKB':    'industry_code',
    'PUFC17_NATEM':  'nature_employment',
    'PUFC18_PNWHRS': 'normal_hours',
    'PUFC19_PHOURS': 'actual_hours',
    'PUFC20_PWMORE': 'want_more_work',
    'PUFC23_PCLASS': 'worker_class',
    'PUFNEWEMPSTAT': 'emp_status',
}

SCHEMA_B = {
    'PUFREG':        'region',
    'PUFURB2015':    'urban_rural',
    'PUFSVYMO':      'survey_month',
    'PUFSVYYR':      'survey_year',
    'PUFHHSIZE':     'hh_size',
    'PUFC01_LNO':    'line_no',
    'PUFC03_REL':    'relationship',
    'PUFC04_SEX':    'sex',
    'PUFC05_AGE':    'age',
    'PUFC06_MSTAT':  'marital_status',
    'PUFC07_GRADE':  'education_grade',
    'PUFC08_CONWR':  'currently_working',
    'PUFC09_WORK':   'worked_last_week',
    'PUFC13_PROCC':  'occupation_code',
    'PUFC15_PKB':    'industry_code',
    'PUFC16_NATEM':  'nature_employment',
    'PUFC17_PNWHRS': 'normal_hours',
    'PUFC18_PHOURS': 'actual_hours',
    'PUFC19_PWMORE': 'want_more_work',
    'PUFC21_PCLASS': 'worker_class',
    'PUFNEWEMPSTAT': 'emp_status',
}

SCHEMA_C = SCHEMA_A.copy()

CANONICAL_COLS = list(SCHEMA_A.values())

`detect_schema` picks the right rename map based on a column that only appears in Schema B. `load_harmonise` loads a single monthly CSV, renames columns to our canonical names, and fills any missing canonical columns with NaN so every file has the same shape.

In [6]:
def detect_schema(cols):
    cols_upper = [c.upper() for c in cols]
    if 'PUFC09_WORK' in cols_upper:
        return SCHEMA_B
    return SCHEMA_A


def load_harmonise(path):
    df = pd.read_csv(path, low_memory=False, dtype=str)
    df.columns = df.columns.str.strip().str.upper()
    schema_map = detect_schema(df.columns)
    rename_map = {k: v for k, v in schema_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)
    for col in CANONICAL_COLS:
        if col not in df.columns:
            df[col] = np.nan
    return df[CANONICAL_COLS].copy()

This loads all 48 monthly CSVs and concatenates them into a single dataframe.

In [7]:
all_paths = sorted(
    set(DATA_DIR.rglob("*.csv")) | set(DATA_DIR.rglob("*.CSV")),
    key=lambda p: p.name.lower()
)

frames = [load_harmonise(p) for p in all_paths]
raw = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(all_paths)} files  |  Total rows: {len(raw):,}')

Loaded 48 files  |  Total rows: 6,554,055


## 2. Type Coercion & Cleaning

All values were loaded as strings to avoid silent numeric truncation of coded values. Now we cast columns to appropriate types.

In [8]:
def to_int_nullable(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors='coerce').astype('Int64')


def to_str_stripped(series):
    s = series.astype(str).str.strip()
    return s.replace({'': np.nan, 'nan': np.nan, 'NaN': np.nan})


INT_COLS = ['survey_month', 'survey_year', 'region', 'hh_size', 'line_no',
            'relationship', 'sex', 'age', 'marital_status',
            'urban_rural', 'currently_working', 'worked_last_week',
            'normal_hours', 'actual_hours', 'want_more_work',
            'worker_class', 'nature_employment', 'emp_status']

for col in INT_COLS:
    raw[col] = to_int_nullable(raw[col])

STR_COLS = ['education_grade', 'occupation_code', 'industry_code']
for col in STR_COLS:
    raw[col] = to_str_stripped(raw[col])

## 3. Filter to Employed Working-Age Population

`emp_status == 1` captures all employed persons as defined by PSA. Age filter ≥ 15 follows PSA’s working-age population definition.

In [9]:
employed = raw[(raw['emp_status'] == 1) & (raw['age'] >= 15)].copy()
print(f'Employed working-age: {len(employed):,} rows')

Employed working-age: 2,740,496 rows


## 4. Construct `is_stagnant` Target Variable

### Definition

Employee stagnation = being stuck in a low-quality labour market position with insufficient upward mobility. No direct survey question exists, so we operationalise it from three PSA-defined indicators. A worker is **stagnant** if ≥ 2 of the following 3 criteria are satisfied:

| Criterion | Variable | Logic |
|-----------|----------|-------|
| **C1 – Visible underemployment** | `want_more_work` | = 1 (wants additional work or hours) |
| **C2 – Precarious job** | `nature_employment`, `worker_class` | nature ∈ {2=temporary, 3=casual/seasonal} OR class = 6 (unpaid family worker) |
| **C3 – Education-occupation mismatch** | `education_grade`, `occupation_code` | College-educated (grade level ≥ 6) working in elementary occupations (PSOC major group = 9) |

### Leakage Prevention

All criterion inputs are dropped from the dataset after target construction — C1 (`want_more_work`), C2 (`nature_employment`, `worker_class`), and C3 (`education_level`, `occupation_major`, plus the raw `education_grade` and `occupation_code` they are derived from). Keeping any of these would give a downstream model a direct path to reconstructing the rule it was trained against.

### Why 2-of-3 composite?

Any single indicator has blind spots. Requiring 2 of 3 signals to co-occur reduces false positives and captures the multi-dimensional nature of genuine stagnation.

In [10]:
employed['education_level'] = (
    employed['education_grade']
    .astype(str).str.strip().str[0]
    .replace({'n': np.nan, 'N': np.nan})
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

employed['occupation_major'] = (
    employed['occupation_code']
    .astype(str).str.strip().str[0]
    .replace({'': np.nan, 'n': np.nan, 'N': np.nan})
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)

In [11]:
c1 = (employed['want_more_work'] == 1).fillna(False).astype(int)

c2 = (
    employed['nature_employment'].isin([2, 3]).fillna(False) |
    (employed['worker_class'] == 6).fillna(False)
).astype(int)

c3 = (
    (employed['education_level'] >= 6).fillna(False) &
    (employed['occupation_major'] == 9).fillna(False)
).astype(int)

employed['is_stagnant'] = ((c1 + c2 + c3) >= 2).astype(int)

print(f'is_stagnant = 1: {employed["is_stagnant"].sum():,}  ({employed["is_stagnant"].mean()*100:.1f}%)')
print(f'is_stagnant = 0: {(employed["is_stagnant"]==0).sum():,}  ({(1-employed["is_stagnant"].mean())*100:.1f}%)')

is_stagnant = 1: 221,394  (8.1%)
is_stagnant = 0: 2,519,102  (91.9%)


## 5. Merge GDP per Person Employed

Year-level macro context from the World Bank (constant 2021 PPP dollars). Joined on `survey_year`.

In [12]:
gdp = pd.read_csv(GDP_PATH)
gdp_phl = (
    gdp[gdp['Code'] == 'PHL']
    [['Year', 'GDP per person employed (constant 2021 PPP $)']]
    .rename(columns={
        'Year': 'survey_year',
        'GDP per person employed (constant 2021 PPP $)': 'gdp_per_employed'
    })
)

employed = employed.merge(gdp_phl, on='survey_year', how='left')
print(f'GDP merge null count: {employed["gdp_per_employed"].isna().sum()}')

GDP merge null count: 0


## 6. Feature Engineering & Criterion Column Drop

Features selected must have no causal overlap with the stagnation criteria. The following are dropped before saving:

- `want_more_work`, `nature_employment`, `worker_class` → C1 and C2 inputs (direct target construction)
- `education_level`, `occupation_major` → C3 inputs (derived from `education_grade` and `occupation_code`)
- `education_grade`, `occupation_code` → raw sources of C3 inputs (prevent indirect reconstruction)

`survey_month` is retained as a cyclical feature to capture seasonality.

In [13]:
employed['industry_major'] = (
    pd.to_numeric(employed['industry_code'].astype(str).str.strip(), errors='coerce')
    .astype('Int64')
)


def psic_sector(code):
    if pd.isna(code): return 0
    c = int(code)
    if c <= 3:  return 1
    if c <= 9:  return 2
    if c <= 33: return 3
    if c <= 43: return 4
    if c <= 47: return 5
    if c <= 56: return 6
    if c <= 66: return 7
    if c <= 82: return 8
    if c <= 88: return 9
    return 10


employed['industry_sector'] = employed['industry_major'].apply(psic_sector)
employed['month_sin'] = np.sin(2 * np.pi * employed['survey_month'].astype(float) / 12)
employed['month_cos'] = np.cos(2 * np.pi * employed['survey_month'].astype(float) / 12)

In [14]:
CRITERION_COLS = [
    'want_more_work',    # C1
    'nature_employment', # C2
    'worker_class',      # C2
    'education_level',   # C3 derived
    'occupation_major',  # C3 derived
    'education_grade',   # raw source of education_level
    'occupation_code',   # raw source of occupation_major
]
employed = employed.drop(columns=CRITERION_COLS, errors='ignore')
print(f'Columns remaining: {list(employed.columns)}')

Columns remaining: ['region', 'urban_rural', 'survey_month', 'survey_year', 'hh_size', 'line_no', 'relationship', 'sex', 'age', 'marital_status', 'currently_working', 'worked_last_week', 'industry_code', 'normal_hours', 'actual_hours', 'emp_status', 'is_stagnant', 'gdp_per_employed', 'industry_major', 'industry_sector', 'month_sin', 'month_cos']


## 7. Save Processed Data

In [15]:
employed.to_csv(OUT_DIR / 'employed_processed.csv', index=False)

size_mb = (OUT_DIR / 'employed_processed.csv').stat().st_size / 1e6
print(f'Saved: employed_processed.csv  ({size_mb:.1f} MB)')
print(f'Rows: {len(employed):,}')
print(f'Stagnation rate: {employed["is_stagnant"].mean()*100:.1f}%')

Saved: employed_processed.csv  (258.1 MB)
Rows: 2,740,496
Stagnation rate: 8.1%
